In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

import warnings
warnings.filterwarnings("ignore")  # suppresses some runtime warnings

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [2]:
casenames = {'cpl_control': 'cpl_1850_f19',
             'cpl_CO2ramp': 'cpl_CO2ramp_f19',
             'som_control': 'som_1850_f19',
             'som_2xCO2':   'som_1850_2xCO2',
            }
# The path to the THREDDS server, should work from anywhere
basepath = 'http://thredds.atmos.albany.edu:8080/thredds/dodsC/CESMA/'
# For better performance if you can access the roselab_rit filesystem (e.g. from JupyterHub)
# basepath = '/roselab_rit/cesm_archive/'
casepaths = {}
for name in casenames:
    casepaths[name] = basepath + casenames[name] + '/concatenated/'

In [3]:
# make a dictionary of all the CAM atmosphere output
atm = {}
for name in casenames:
    path = casepaths[name] + casenames[name] + '.cam.h0.nc'
    print('Attempting to open the dataset ', path)
    atm[name] = xr.open_dataset(path, decode_times=False)

Attempting to open the dataset  http://thredds.atmos.albany.edu:8080/thredds/dodsC/CESMA/cpl_1850_f19/concatenated/cpl_1850_f19.cam.h0.nc


NameError: name 'xr' is not defined

In [4]:
# note: times are days since beginning, stepped by month lengths
atm['cpl_control'].FSNT

# are weights the same for the different sims? yes
# todo: could use itertools.product for fun
for name in casenames:
    for compare_name in casenames:
#         print(atm[name].gw.all() == atm[compare_name].gw.all())
        pass
        
gw = atm['cpl_control'].gw
days_per_year = 365
# print(gw)

KeyError: 'cpl_control'

In [5]:
# define a function to compute the global mean using correctly spatial weights
def global_mean(field, weight=gw):
    return (field * weight).mean(dim=('lat', 'lon')) / weight.mean(dim='lat')

NameError: name 'gw' is not defined

In [6]:
# global_mean(atm['cpl_control'].FSNT)

In [7]:
# global_mean(atm['cpl_control']['FLNT'])

In [8]:
# make time-series of global mean FSNT and FLNT
fluxes_global = {}

In [9]:
model_types = ['cpl', 'som']  # coupled or slab ocean models
radiative_fluxes = {'FSNT': 'ASR', 'FLNT': 'OLR'}

for flux in radiative_fluxes.keys():  # iterate over FSNT, FLNT
    print(radiative_fluxes[flux])
    fluxes_global[radiative_fluxes[flux]] = {}  # remap to ASR and OLR
    for name in casenames:
        # compute global mean for each case:
        fluxes_global[radiative_fluxes[flux]][name] = global_mean(atm[name][flux])
 

ASR


NameError: name 'global_mean' is not defined

In [10]:
print(fluxes_global.keys())
print(fluxes_global['ASR'].keys())
print(f"ASR array:\n\t{fluxes_global['ASR']['cpl_control']}\n")  # show the nested structure
print(f"OLR array:\n\t{fluxes_global['OLR']['cpl_control']}\n")

dict_keys(['ASR'])
dict_keys([])


KeyError: 'cpl_control'

In [11]:
fig, axes = plt.subplots(2,2,figsize=(10,8), dpi=150)
for ii, flux in enumerate(radiative_fluxes):
    for name in casenames:
#         print(f"currently iterating over {flux},\n\tcase {name}")
        if 'cpl' in name:
            ax = axes[0][ii]
            ax.set_title(f'Fully coupled ocean {radiative_fluxes[flux]}')
        else:
            ax = axes[1][ii]
            ax.set_title(f'Slab ocean {radiative_fluxes[flux]}')
        field = fluxes_global[radiative_fluxes[flux]][name]
        field_running = field.rolling(time=12, center=True).mean()
        line = ax.plot(field.time / days_per_year, 
                       field, 
                       label=name,
                       linewidth=0.75,
                       )
        ax.plot(field_running.time / days_per_year, 
                field_running, 
                color=line[0].get_color(),
                linewidth=2,
               )
        
    counter = 0
    for ax in axes:
        ax[ii].legend();
        if counter == 1:
            ax[ii].set_xlabel('Years')
        ax[ii].set_ylabel(f'{flux} (W/m2)')
        ax[ii].grid();
        ax[ii].set_xlim(0,100)
        ax[ii].set_ylim(224.0,242.0)  # make this a function of the range, make it smarter
        counter+=1
# plt.tight_layout()  # not working with suptitle
fig.suptitle('Global mean ASR and OLR in CESM simulations', fontsize=16);

NameError: name 'plt' is not defined

In [12]:
fig, axes = plt.subplots(2,1,figsize=(10,8), dpi=150)
for name in casenames:
    if 'cpl' in name:
        ax = axes[0]
        ax.set_title('Fully coupled ocean')
    else:
        ax = axes[1]
        ax.set_title('Slab ocean')
    field = np.subtract(fluxes_global['ASR'][name], fluxes_global['OLR'][name])  # ASR - OLR
#     print(field)
    field_running = field.rolling(time=12, center=True).mean()
    line = ax.plot(field.time / days_per_year, 
                   field, 
                   label=name,
                   linewidth=0.75,
                   )
    ax.plot(field_running.time / days_per_year, 
            field_running, 
            color=line[0].get_color(),
            linewidth=2,
           )
counter = 0
for ax in axes:
    ax.legend();
    if counter == 1:
        ax.set_xlabel('Years')
    ax.set_ylabel('(ASR - OLR) (W/m2)')
    ax.grid();
    ax.set_xlim(0,100)
    counter+=1
# plt.tight_layout()  # not working with suptitle
fig.suptitle('Global mean net downward energy flux\nat the top of the model in CESM simulations', fontsize=16);

NameError: name 'plt' is not defined

In [13]:
# todo: time average, preserving lat-long
print("ASR time averages")
# extract the last 10 years from the slab ocean control simulation
# and the last 20 years from the coupled control
nyears_slab = 10
nyears_cpl = 20
clim_slice_slab = slice(-(nyears_slab*12),None)
clim_slice_cpl = slice(-(nyears_cpl*12),None)
# extract the last 10 years from the slab ocean control simulation
asr0_slab = fluxes_global['ASR']['som_control'].isel(time=clim_slice_slab).mean(dim='time')
print(f"slab model control: {asr0_slab}")
# extract the last 10 years from the slab 2xCO2 simulation
asr2x_slab = fluxes_global['ASR']['som_2xCO2'].isel(time=clim_slice_slab).mean(dim='time')
print(f"slab model doulbing: {asr2x_slab}")
# and the last 20 years from the coupled control
asr0_cpl = fluxes_global['ASR']['cpl_control'].isel(time=clim_slice_cpl).mean(dim='time')
print(f"coupled model control: {asr0_cpl}")
# extract the last 20 years from the coupled CO2 ramp simulation
asr2x_cpl = fluxes_global['ASR']['cpl_CO2ramp'].isel(time=clim_slice_cpl).mean(dim='time')
print(f"coupled model doubling: {asr2x_cpl}")

ASR time averages


KeyError: 'som_control'

In [14]:
print("OLR time averages")
# extract the last 10 years from the slab ocean control simulation
olr0_slab = fluxes_global['OLR']['som_control'].isel(time=clim_slice_slab).mean(dim='time')
print(f"slab model control: {olr0_slab}")
# extract the last 10 years from the slab 2xCO2 simulation
olr2x_slab = fluxes_global['OLR']['som_2xCO2'].isel(time=clim_slice_slab).mean(dim='time')
print(f"slab model doulbing: {olr2x_slab}")
# and the last 20 years from the coupled control
olr0_cpl = fluxes_global['OLR']['cpl_control'].isel(time=clim_slice_cpl).mean(dim='time')
print(f"coupled model control: {olr0_cpl}")
# extract the last 20 years from the coupled CO2 ramp simulation
olr2x_cpl = fluxes_global['OLR']['cpl_CO2ramp'].isel(time=clim_slice_cpl).mean(dim='time')
print(f"coupled model doubling: {olr2x_cpl}")

OLR time averages


KeyError: 'OLR'

In [15]:
(asr2x_slab - olr2x_slab) - (asr0_slab - olr0_slab)

NameError: name 'asr2x_slab' is not defined

In [16]:
(asr2x_cpl - olr2x_cpl) - (asr0_cpl - olr0_cpl)

NameError: name 'asr2x_cpl' is not defined

In [17]:
(fluxes_global['ASR']['som_2xCO2'] - fluxes_global['OLR']['som_2xCO2']).rolling(time=12, center=True).mean()[-6]

KeyError: 'som_2xCO2'

In [18]:
# compute the annual average imbalance at the end of the coupled simulation
(fluxes_global['ASR']['cpl_CO2ramp'] - fluxes_global['OLR']['cpl_CO2ramp']).rolling(time=12, center=True).mean()[-6]

KeyError: 'cpl_CO2ramp'

In [19]:
# this averages over the last 20 years
global_asr_CO2ramp = atm['cpl_CO2ramp'].FSNT.isel(time=clim_slice_cpl).mean(dim='time')
global_asr_control = atm['cpl_control'].FSNT.isel(time=clim_slice_cpl).mean(dim='time')
delta_asr_cpl = global_asr_CO2ramp - global_asr_control

KeyError: 'cpl_CO2ramp'

In [20]:
# The map projection capabilities come from the cartopy package. There are many possible projections
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [21]:
def make_map(field, title=""):
    '''input field should be a 2D xarray.DataArray on a lat/lon grid.
        Make a filled contour plot of the field, and a line plot of the zonal mean
    '''
    fig = plt.figure(figsize=(14,6), dpi=150)
    nrows = 10; ncols = 3
    mapax = plt.subplot2grid((nrows,ncols), (0,0), colspan=ncols-1, rowspan=nrows-1, projection=ccrs.Robinson())
    barax = plt.subplot2grid((nrows,ncols), (nrows-1,0), colspan=ncols-1)
    plotax = plt.subplot2grid((nrows,ncols), (0,ncols-1), rowspan=nrows-1)
    # add cyclic point so cartopy doesn't show a white strip at zero longitude
    wrap_data, wrap_lon = add_cyclic_point(field.values, coord=field.lon, axis=field.dims.index('lon'))
    cx = mapax.contourf(wrap_lon, field.lat, wrap_data, transform=ccrs.PlateCarree())
    mapax.set_global(); mapax.coastlines();
    plt.colorbar(cx, cax=barax, orientation='horizontal')
    plotax.plot(field.mean(dim='lon'), field.lat)
    plotax.set_ylabel('Latitude')
    plotax.grid()
    fig.suptitle(title, fontsize=16)
    return fig, (mapax, plotax, barax), cx

In [22]:
fig, axes, cx = make_map(delta_asr_cpl,
                        title='Absorbed Shortwave Radiation Anomaly (coupled transient) (W/m2)')

NameError: name 'delta_asr_cpl' is not defined

In [23]:
global_asr_clear_CO2ramp = atm['cpl_CO2ramp'].FSNTC.isel(time=clim_slice_cpl).mean(dim='time')
global_asr_clear_control = atm['cpl_control'].FSNTC.isel(time=clim_slice_cpl).mean(dim='time')
delta_asr_clear_cpl = global_asr_clear_CO2ramp - global_asr_clear_control

KeyError: 'cpl_CO2ramp'

In [24]:
fig, axes, cx = make_map(delta_asr_clear_cpl,
                        title='Absorbed Shortwave Radiation Anomaly (coupled transient) (W/m2)')

NameError: name 'delta_asr_clear_cpl' is not defined